<a href="https://colab.research.google.com/github/danish2k04/Getting-into-Pytorch-Deep-Learning/blob/main/2_pap_cell_ConvNeXt_Tiny.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pandas scikit-learn timm

import os, random, numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from torchvision import models, transforms
import timm

In [2]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

from google.colab import drive
drive.mount('/content/drive')

ROOT      = "/content/drive/MyDrive/pap_cell_project/data"
TRAIN_CSV = os.path.join(ROOT, "isbi2025-ps3c-train-dataset.csv")

Using device: cuda
Mounted at /content/drive


In [3]:
!apt-get install -y p7zip-full -q

Reading package lists...
Building dependency tree...
Reading state information...
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


In [4]:
DATA_DIR  = "/content/apacc_data"
TRAIN_DIR = DATA_DIR
os.makedirs(DATA_DIR, exist_ok=True)
!7z x "{os.path.join(ROOT, 'isbi2025-ps3c-train-dataset.7z')}" -o{DATA_DIR} -y


7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/drive/MyDrive/pap_cell_project/data/                                                       1 file, 16087644759 bytes (15 GiB)

Extracting archive: /content/drive/MyDrive/pap_cell_project/data/isbi2025-ps3c-train-dataset.7z
--
Path = /content/drive/MyDrive/pap_cell_project/data/isbi2025-ps3c-train-dataset.7z
Type = 7z
Physical Size = 16087644759
Headers Size = 1272355
Method = LZMA2:26
Solid = +
Blocks = 1

  0%      0% 65 - unhealthy/isbi2025_ps3c_train_image_82780.png                                                       

In [5]:
DROP_BOTHCELLS  = True
IMG_SIZE        = 384          # ConvNeXt native resolution — better than 300
BATCH_SIZE      = 24           # smaller batch for larger images
EPOCHS          = 40
CHECKPOINT_PATH = os.path.join(ROOT, "checkpoint_convnext.pth")
MODEL_SAVE_PATH = os.path.join(ROOT, "best_model_convnext.pth")

In [6]:
class PadToSquareWhite:
    def __call__(self, img):
        w, h = img.size
        m = max(w, h)
        l = (m - w) // 2
        t = (m - h) // 2
        return ImageOps.expand(img, border=(l, t, m - w - l, m - h - t), fill=(255, 255, 255))

train_tf = transforms.Compose([
    PadToSquareWhite(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30, fill=255),          # was 20
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), fill=255),  # new — random shift
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05),  # stronger
    transforms.RandomGrayscale(p=0.05),               # new — occasional grayscale
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),  # new — random erase patches
])

eval_tf = transforms.Compose([
    PadToSquareWhite(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

In [7]:
df = pd.read_csv(TRAIN_CSV)
if DROP_BOTHCELLS:
    df = df[df["label"] != "bothcells"].copy()

classes      = sorted(df["label"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
df["target"] = df["label"].map(class_to_idx)

print("Classes:", classes)
print("Class → index:", class_to_idx)
print("Label distribution:\n", df["label"].value_counts())

Classes: ['healthy', 'rubbish', 'unhealthy']
Class → index: {'healthy': 0, 'rubbish': 1, 'unhealthy': 2}
Label distribution:
 label
rubbish      50371
healthy      28895
unhealthy     2366
Name: count, dtype: int64


In [8]:
class PapCellDataset(Dataset):
    def __init__(self, df, img_root, tfm):
        self.df       = df.reset_index(drop=True)
        self.img_root = img_root
        self.tfm      = tfm

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, row["label"], row["image_name"])
        img      = Image.open(img_path).convert("RGB")
        return self.tfm(img), int(row["target"])

In [9]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_idx, val_idx = next(sss.split(df["image_name"], df["target"]))
df_train = df.iloc[train_idx].copy()
df_val   = df.iloc[val_idx].copy()

train_ds = PapCellDataset(df_train, TRAIN_DIR, train_tf)
val_ds   = PapCellDataset(df_val,   TRAIN_DIR, eval_tf)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

Train: 69387 | Val: 12245


In [10]:
counts        = df_train["target"].value_counts().sort_index().values.astype(np.float32)
class_weights = 1.0 / counts                        # full inverse frequency
class_weights = class_weights / class_weights.mean()

sample_weights = df_train["target"].map(
    {i: w for i, w in enumerate(class_weights)}
).values

sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True, persistent_workers=True)

print("Class weights:", dict(zip(classes, class_weights.round(3))))

Class weights: {'healthy': np.float32(0.218), 'rubbish': np.float32(0.125), 'unhealthy': np.float32(2.658)}


### CELL 11 — Model (ConvNeXt-Tiny)
 Why ConvNeXt-Tiny:
   - Outperforms EfficientNet-B4 on most medical imaging benchmarks
   - Better gradient flow than EfficientNet for small minority classes
  - Faster per epoch than EfficientNet-B4 on T4

In [11]:
model = timm.create_model("convnext_tiny", pretrained=True, num_classes=len(classes))
model = model.to(device)

# Back to CrossEntropyLoss — sampler handles imbalance, loss stays stable
criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32, device=device),
    label_smoothing=0.05
)

# Single LR for everything — layered LR was overcomplicating things
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler(enabled=(device == "cuda"))

best_state, best_macro_f1 = None, -1.0
print("Model ready.")

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Model: ConvNeXt-Tiny | Params: 27.0 M


In [12]:
def run_epoch(loader, train):
    model.train(train)
    all_y, all_p, total_loss = [], [], 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in tqdm(loader, desc="train" if train else "val ", leave=False):
            x, y = x.to(device), y.to(device)
            if train:
                optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda", enabled=(device == "cuda")):
                logits = model(x)
                loss   = criterion(logits, y)
            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * x.size(0)
            all_y.append(y.cpu().numpy())
            all_p.append(logits.argmax(1).cpu().numpy())

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_p)
    return (
        total_loss / len(y_true),
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average="macro"),
        f1_score(y_true, y_pred, average="weighted"),
        y_true,
        y_pred,
    )



In [ ]:
import os
ckpt = os.path.join(ROOT, "checkpoint_convnext.pth")
if os.path.exists(ckpt):
    os.remove(ckpt)
    print("Old checkpoint deleted — starting fresh.")

In [13]:
start_epoch      = 1
patience         = 7
patience_counter = 0

if os.path.exists(CHECKPOINT_PATH):
    print(f"Checkpoint found — resuming from {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    scaler.load_state_dict(ckpt["scaler_state"])
    best_state       = ckpt["best_state"]
    best_macro_f1    = ckpt["best_macro_f1"]
    start_epoch      = ckpt["epoch"] + 1
    patience_counter = ckpt["patience_counter"]
    print(f"Resuming from epoch {start_epoch} | best macro-F1 so far: {best_macro_f1:.4f}")
else:
    print("No checkpoint — starting fresh.")

for epoch in range(start_epoch, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader,   train=False)
    scheduler.step()

    print(f"Epoch {epoch:02d} | "
          f"train_loss={tr[0]:.4f} | val_loss={va[0]:.4f} | "
          f"val_acc={va[1]:.4f} | val_macro_f1={va[2]:.4f} | val_weighted_f1={va[3]:.4f}")

    if va[2] > best_macro_f1:
        best_macro_f1    = va[2]
        best_state       = {k: v.cpu() for k, v in model.state_dict().items()}
        patience_counter = 0
        torch.save(best_state, MODEL_SAVE_PATH)
        print(f"  ✓ New best macro-F1: {best_macro_f1:.4f} — saved.")
    else:
        patience_counter += 1

    torch.save({
        "epoch":            epoch,
        "model_state":      model.state_dict(),
        "optimizer_state":  optimizer.state_dict(),
        "scheduler_state":  scheduler.state_dict(),
        "scaler_state":     scaler.state_dict(),
        "best_state":       best_state,
        "best_macro_f1":    best_macro_f1,
        "patience_counter": patience_counter,
    }, CHECKPOINT_PATH)

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}.")
        break

print(f"\nTraining done. Best val macro-F1: {best_macro_f1:.4f}")


No checkpoint — starting fresh.


train:   0%|          | 0/2892 [00:00<?, ?it/s]

val :   0%|          | 0/511 [00:00<?, ?it/s]

Epoch 01 | train_loss=0.2963 | val_loss=1.3262 | val_acc=0.5094 | val_macro_f1=0.4652 | val_weighted_f1=0.6271
  ✓ New best macro-F1: 0.4652 — saved.


train:   0%|          | 0/2892 [00:00<?, ?it/s]

...

val :   0%|          | 0/511 [00:00<?, ?it/s]

Epoch 16 | train_loss=0.1938 | val_loss=1.1899 | val_acc=0.7902 | val_macro_f1=0.6480 | val_weighted_f1=0.8234


train:   0%|          | 0/2892 [00:00<?, ?it/s]

val :   0%|          | 0/511 [00:00<?, ?it/s]

Epoch 17 | train_loss=0.1948 | val_loss=1.1497 | val_acc=0.8241 | val_macro_f1=0.6828 | val_weighted_f1=0.8412
Early stopping at epoch 17.

Training done. Best val macro-F1: 0.6849


In [14]:
model = timm.create_model("convnext_tiny", pretrained=False, num_classes=len(classes))
model = model.to(device)
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model.eval()
print("Best model loaded.")

va = run_epoch(val_loader, train=False)
print("\nValidation Report:")
print(classification_report(va[4], va[5], target_names=classes, digits=4))
print("Confusion Matrix:\n", confusion_matrix(va[4], va[5]))

Best model loaded.


val :   0%|          | 0/511 [00:00<?, ?it/s]


Validation Report:
              precision    recall  f1-score   support

     healthy     0.7906    0.9077    0.8451      4334
     rubbish     0.9692    0.8029    0.8783      7556
   unhealthy     0.2240    0.6366    0.3314       355

    accuracy                         0.8352     12245
   macro avg     0.6612    0.7824    0.6849     12245
weighted avg     0.8844    0.8352    0.8507     12245

Confusion Matrix:
 [[3934  166  234]
 [ 940 6067  549]
 [ 102   27  226]]


In [15]:
TEST_DATA_DIR = "/content/apacc_test_data"
os.makedirs(TEST_DATA_DIR, exist_ok=True)
!7z x "{os.path.join(ROOT, 'isbi2025-ps3c-test-dataset.7z')}" -o{TEST_DATA_DIR} -y
print("Test images extracted.")


7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/drive/MyDrive/pap_cell_project/data/                                                       1 file, 3213844601 bytes (3065 MiB)

Extracting archive: /content/drive/MyDrive/pap_cell_project/data/isbi2025-ps3c-test-dataset.7z
--
Path = /content/drive/MyDrive/pap_cell_project/data/isbi2025-ps3c-test-dataset.7z
Type = 7z
Physical Size = 3213844601
Headers Size = 275513
Method = LZMA2:26
Solid = +
Blocks = 1

  0%      1% 224 - isbi2025_ps3c_test_image_00235.png                                               2% 431 - isbi2025_ps3c_test_image_0

In [16]:
test_csv_candidates = [
    os.path.join(ROOT, "isbi2025-ps3c-test-dataset-annotated.csv"),
    os.path.join(ROOT, "isbi2025-ps3c-test-dataset.csv"),
    "isbi2025-ps3c-test-dataset-annotated.csv",
]
TEST_CSV_PATH = None
for p in test_csv_candidates:
    if os.path.exists(p):
        TEST_CSV_PATH = p
        print(f"Found test CSV: {p}")
        break

if TEST_CSV_PATH is None:
    raise FileNotFoundError("No test CSV found in Drive ROOT folder.")

df_test = pd.read_csv(TEST_CSV_PATH)
if DROP_BOTHCELLS:
    df_test = df_test[df_test["label"] != "bothcells"].copy()
df_test["target"] = df_test["label"].map(class_to_idx)

print(f"Test samples: {len(df_test)}")
print(df_test["label"].value_counts())

Found test CSV: /content/drive/MyDrive/pap_cell_project/data/isbi2025-ps3c-test-dataset-annotated.csv
Test samples: 18159
label
rubbish      11757
healthy       5826
unhealthy      576
Name: count, dtype: int64


In [17]:
class PapCellTestDataset(Dataset):
    def __init__(self, df, img_root, tfm):
        self.df       = df.reset_index(drop=True)
        self.img_root = img_root
        self.tfm      = tfm

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, row["image_name"])
        img      = Image.open(img_path).convert("RGB")
        return self.tfm(img), int(row["target"])

test_ds     = PapCellTestDataset(df_test, TEST_DATA_DIR, eval_tf)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)
print(f"Test samples loaded: {len(test_ds)}")

Test samples loaded: 18159


In [18]:
model.eval()
te = run_epoch(test_loader, train=False)

print("\nTest Report (healthy / rubbish / unhealthy):")
print(classification_report(te[4], te[5], target_names=classes, digits=4))
print("Confusion Matrix:\n", confusion_matrix(te[4], te[5]))
print(f"\nTest macro-F1:    {te[2]:.4f}")
print(f"Test weighted-F1: {te[3]:.4f}")
print(f"Test accuracy:    {te[1]:.4f}")

val :   0%|          | 0/757 [00:00<?, ?it/s]


Test Report (healthy / rubbish / unhealthy):
              precision    recall  f1-score   support

     healthy     0.7279    0.8953    0.8030      5826
     rubbish     0.9642    0.7590    0.8494     11757
   unhealthy     0.1749    0.5278    0.2627       576

    accuracy                         0.7954     18159
   macro avg     0.6223    0.7274    0.6384     18159
weighted avg     0.8634    0.7954    0.8159     18159

Confusion Matrix:
 [[5216  260  350]
 [1749 8924 1084]
 [ 201   71  304]]

Test macro-F1:    0.6384
Test weighted-F1: 0.8159
Test accuracy:    0.7954


### CELL 18 — TTA Evaluation on Test Set
TTA = run each image through 5 augmented versions + original,
average the softmax probabilities, then take argmax.
Much safer than threshold tuning — no risk of class collapse.

In [19]:
tta_tf = transforms.Compose([
    PadToSquareWhite(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15, fill=255),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

def run_tta(loader, n_aug=5):
    model.eval()
    all_y, all_probs = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc="TTA", leave=False):
            x = x.to(device)
            # original pass
            with torch.amp.autocast(device_type="cuda", enabled=(device == "cuda")):
                probs = torch.softmax(model(x), dim=1)
            # augmented passes
            for _ in range(n_aug - 1):
                x_aug = torch.stack([
                    tta_tf(transforms.ToPILImage()(xi.cpu())) for xi in x
                ]).to(device)
                with torch.amp.autocast(device_type="cuda", enabled=(device == "cuda")):
                    probs += torch.softmax(model(x_aug), dim=1)
            probs /= n_aug
            all_probs.append(probs.cpu().numpy())
            all_y.append(y.numpy())

    y_true = np.concatenate(all_y)
    y_prob = np.concatenate(all_probs)
    y_pred = np.argmax(y_prob, axis=1)
    return y_true, y_pred

y_true, y_pred = run_tta(test_loader, n_aug=5)

print("\nTTA Test Report (healthy / rubbish / unhealthy):")
print(classification_report(y_true, y_pred, target_names=classes, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
print(f"\nTest macro-F1:    {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Test weighted-F1: {f1_score(y_true, y_pred, average='weighted'):.4f}")
print(f"Test accuracy:    {accuracy_score(y_true, y_pred):.4f}")

TTA:   0%|          | 0/757 [00:00<?, ?it/s]


TTA Test Report (healthy / rubbish / unhealthy):
              precision    recall  f1-score   support

     healthy     0.3542    0.0292    0.0539      5826
     rubbish     0.6632    0.9292    0.7740     11757
   unhealthy     0.2091    0.4375    0.2830       576

    accuracy                         0.6249     18159
   macro avg     0.4088    0.4653    0.3703     18159
weighted avg     0.5496    0.6249    0.5274     18159

Confusion Matrix:
 [[  170  5246   410]
 [  289 10925   543]
 [   21   303   252]]

Test macro-F1:    0.3703
Test weighted-F1: 0.5274
Test accuracy:    0.6249
